# **Objective:**

Learn how prompt engineering can be used to guide a Generative AI model (Gemini) to perform, explain, and reason about **data cleaning tasks** — without writing manual data-cleaning code ourselves. Each prompt targets a different real-world messy-data scenario and a different prompting technique.

Python
→ google-genai
→ Gemini API
→ Gemini Model

1. Install Library

        ↓
pip install google-genai

2. Import Library

        ↓
from google import genai

3. Create Client

        ↓
client = genai.Client(api_key)

4. Select Model

        ↓
gemini-2.5-flash

5. Send Prompt

        ↓
generate_content()

6. Receive Response

        ↓
response.text

7. Display Output

        ↓
print()

# 1. Generate API Key

To generate an API key go to: https://aistudio.google.com/

# https://aistudio.google.com/api-keys

# 2. Install Library

In [1]:
!pip install -U google-genai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.4/56.4 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 258.6/258.6 kB 11.0 MB/s eta 0:00:00
  Attempting uninstall: google-auth
    Found existing installation: google-auth 2.49.0
    Uninstalling google-auth-2.49.0:
      Successfully uninstalled google-auth-2.49.0
  Attempting uninstall: google-genai
    Found existing installation: google-genai 2.11.0
    Uninstalling google-genai-2.11.0:
      Successfully uninstalled google-genai-2.11.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.56.2 which is incompatible.


# 3. Import Library

In [2]:
from google import genai

# 4. Create Client

In [3]:
from dotenv import load_dotenv
import os
from google import genai

load_dotenv()

api_key = os.getenv("GEMINI_API_KEY")
client = genai.Client(api_key=api_key)

# 5. Select Model

In [5]:
import google.generativeai as genai_check

genai_check.configure(api_key=api_key)

for m in genai_check.list_models():
    if "generateContent" in m.supported_generation_methods:
        print(m.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-3.6-flash
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-

# 6. Finalized model: gemini-2.5-flash ( Better Reasoning )

In [7]:
response = client.models.generate_content(
    model="gemini-3.5-flash",
    contents="Introduce yourself as a data cleaning assistant in one line."
)

print(response.text)

Hi! I’m your data cleaning assistant, here to scrub away duplicates, fix inconsistencies, and transform your messy datasets into pristine, analysis-ready assets.


# 7. Data Cleaning Prompt Experiments (20 Prompts)

Each experiment below sends a **prompt describing a messy data scenario** to Gemini and captures the **model's response** as the result. No manual cleaning code is written — the model performs/explains the cleaning task directly. Each section also names the **prompting technique** being demonstrated.

# 7.1: Experiment – Zero-Shot Missing Value Handling

**Prompting Technique:** Zero-Shot Prompting

**Formula/Template:** Task + Raw Data

In [8]:
prompt = """
Here is a dataset of customer records with missing values:

id, name, age, city
1, Alice, 28, Mumbai
2, Bob, , Delhi
3, Charlie, 35,
4, David, , Pune

Identify the missing values in each column and suggest the best way to handle each one (e.g., imputation, removal), with reasoning.
"""

response = client.models.generate_content(
    model="gemini-3.5-flash",
    contents=prompt
)

print(response.text)

Based on the dataset provided, here is the identification of the missing values and the recommended strategies for handling them.

---

### **1. Identification of Missing Values**
*   **`id`**: No missing values.
*   **`name`**: No missing values.
*   **`age`** (Numerical): Missing for **Bob** (ID 2) and **David** (ID 4).
*   **`city`** (Categorical): Missing for **Charlie** (ID 3).

---

### **2. Recommended Strategy for Each Column**

#### **Column: `age` (Numerical)**
*   **Identified Missing Values:** Bob (Row 2) and David (Row 4).
*   **Suggested Action:** **Median Imputation** (or Mean Imputation).
*   **Reasoning:** 
    *   **Why not deletion?** Removing these rows would mean deleting 50% of your entire dataset (2 out of 4 records), which results in a massive loss of information.
    *   **Why Median/Mean?** For numerical features like age, filling missing values with the average (mean) or middle value (median) of the existing data is a standard practice. 
    *   *Calculation:

Observation: Without extra guidance, the model identifies missing fields but may give generic advice rather than context-aware suggestions.

# 7.2: Experiment – Role-Based Prompting for Duplicate Detection

**Prompting Technique:** Role-Based Prompting

**Formula/Template:** Role + Task + Data

In [9]:
prompt = """
You are a senior data quality analyst reviewing a customer database before a migration.

Dataset:
101, John Smith, john@mail.com
102, J. Smith, john@mail.com
103, Priya Rao, priya@mail.com
104, John Smith, john@mail.com

Identify likely duplicate records, explain why they are considered duplicates, and recommend which record to keep.
"""

response = client.models.generate_content(
    model="gemini-3.5-flash",
    contents=prompt
)

print(response.text)

**MEMORANDUM**

**TO:** Migration Project Team  
**FROM:** Senior Data Quality Analyst  
**DATE:** October 24, 2023  
**SUBJECT:** Pre-Migration Data Quality Assessment: Customer Duplicate Analysis  

---

### 1. Executive Summary
During the pre-migration profiling of the legacy customer database, I identified a high-probability duplicate cluster involving three out of the four analyzed records. Failing to resolve these before migration will result in bloated database sizes, skewed marketing analytics, and a poor customer experience (e.g., sending duplicate communications). 

---

### 2. Duplicate Identification & Technical Analysis

Based on our deterministic and probabilistic matching algorithms, we have identified **Records 101, 102, and 104** as duplicates representing the same physical individual. 

| Customer ID | Name | Email | Duplicate Type | Analysis & Confidence Level |
| :--- | :--- | :--- | :--- | :--- |
| **101** | John Smith | `john@mail.com` | *Base/Golden Candidate* | 

Observation: Assigning the 'senior data quality analyst' role makes the response more structured, professional, and decision-oriented.

# 7.3: Experiment – Constraint Prompting for Outlier Explanation

**Prompting Technique:** Constraint Prompting

**Formula/Template:** Task + Constraints

In [10]:
prompt = """
Explain how to detect outliers in this list of transaction amounts in under 100 words:
[45, 52, 48, 51, 4500, 49, 53, 47]
"""

response = client.models.generate_content(
    model="gemini-3.5-flash",
    contents=prompt
)

print(response.text)

To detect the outlier in this list, use the **Interquartile Range (IQR) method**:

1. **Sort the data**: 45, 47, 48, 49, 51, 52, 53, 4500.
2. **Find Quartiles**: Q1 (25th percentile) is 47.5; Q3 (75th percentile) is 52.5.
3. **Calculate IQR**: $Q3 - Q1 = 5$.
4. **Determine Threshold**: The upper limit for normal data is $Q3 + (1.5 \times IQR) = 60$.

Since **4500** is far above 60, it is mathematically flagged as the outlier. Visually, it also clearly deviates from the \$45–\$53 norm.


Observation: The response becomes concise and beginner-friendly while still covering the key outlier-detection idea.

# 7.4: Experiment – Structured Output for Data Type Correction

**Prompting Technique:** Structured Output Prompting

**Formula/Template:** Task + Format

In [11]:
prompt = """
Review this table and list the correct data type each column should be, in a table with columns: Column, Current Issue, Recommended Type.

order_id, order_date, quantity, price
"1001", "2024/13/40", "5", "$19.99"
"1002", "2024-01-15", "-2", "20"
"""

response = client.models.generate_content(
    model="gemini-3.5-flash",
    contents=prompt
)

print(response.text)

Here is the review of your table's data types, along with the identified issues and recommended types:

| Column | Current Issue | Recommended Type |
| :--- | :--- | :--- |
| **order_id** | Stored as text/string (indicated by quotation marks) instead of a numeric identifier. | **INT** (Integer) |
| **order_date** | Stored as text with inconsistent formatting (slashes vs. hyphens). It also contains a corrupt/invalid date value (`"2024/13/40"` has an impossible month 13 and day 40). | **DATE** (standardized to `YYYY-MM-DD` format) |
| **quantity** | Stored as text. It also contains a negative value (`"-2"`), which may be invalid for a standard purchase order unless it represents a refund/return. | **INT** (or **UNSIGNED INT** if negative values are not allowed by your business logic) |
| **price** | Stored as text, contains currency symbols (`$`) which prevent mathematical calculations, and has inconsistent decimal formats (`"$19.99"` vs `"20"`). | **DECIMAL(10,2)** (to ensure precise cu

Observation: Explicitly requesting a table format produces an organized, easy-to-scan response instead of a paragraph.

# 7.5: Experiment – Few-Shot Prompting for Categorical Standardization

**Prompting Technique:** Few-Shot Prompting

**Formula/Template:** Examples + Task

In [12]:
prompt = """
Standardize inconsistent category values using the pattern shown below:

Example 1: "male", "MALE", "Male " -> "Male"
Example 2: "female", "FEMALE", "Female " -> "Female"

Now standardize this list the same way:
["M", "m", "Male", "FEMALE", "f", "female "]
"""

response = client.models.generate_content(
    model="gemini-3.5-flash",
    contents=prompt
)

print(response.text)

Based on the patterns provided, the standardized list is:

**["Male", "Male", "Male", "Female", "Female", "Female"]**


Observation: Providing examples first (few-shot) helps the model infer the exact normalization pattern expected, even for unseen abbreviations like 'M' or 'f'.

# 7.6: Experiment – Chain-of-Thought Outlier Reasoning

**Prompting Technique:** Chain-of-Thought Prompting

**Formula/Template:** Task + 'Think step by step'

In [13]:
prompt = """
Think step by step and show your reasoning before giving the final answer.

Dataset of ages: [22, 25, 24, 23, 150, 26, 24, -5]

Step 1: Identify which values are statistically or logically implausible.
Step 2: Explain why each flagged value is problematic.
Step 3: Give a final list of clean, valid ages.
"""

response = client.models.generate_content(
    model="gemini-3.5-flash",
    contents=prompt
)

print(response.text)

Here is the step-by-step reasoning and final dataset based on your request.

### **Step 1: Identify which values are statistically or logically implausible**
Looking at the dataset `[22, 25, 24, 23, 150, 26, 24, -5]`, the flagged values are:
* **`150`**
* **`-5`**

---

### **Step 2: Explain why each flagged value is problematic**
* **`150` (Logically and Statistically Implausible):** The maximum verified human lifespan in recorded history is 122 years (Jeanne Calment). An age of 150 is biologically impossible for a human and is also a extreme statistical outlier compared to the rest of the dataset (which clusters around the mid-20s). This is likely a data entry error (for example, typing "150" instead of "15" or "50").
* **`-5` (Logically Impossible):** Age cannot be a negative number. Time lived must be greater than or equal to zero. This is a clear data entry error (possibly a typo where a hyphen was mistaken for a minus sign, or the age was meant to be "5").

---

### **Step 3: Fin

Observation: Asking for step-by-step reasoning produces a transparent, auditable cleaning decision rather than a black-box answer.

# 7.7: Experiment – Prompt Refinement for Whitespace/Formatting Cleanup

**Prompting Technique:** Prompt Refinement (Vague vs Refined)

**Formula/Template:** Vague Prompt -> Refined Prompt

In [14]:
prompt = """
Initial (vague) prompt:
Clean this text: [" New York ", "new york", "NEW YORK", "New York City"]

Refined prompt:
Clean and standardize this list of city names to a single consistent format (Title Case, trimmed whitespace, and merge "New York City" into "New York" since they refer to the same city):
[" New York ", "new york", "NEW YORK", "New York City"]

Run the refined prompt and show the result.
"""

response = client.models.generate_content(
    model="gemini-3.5-flash",
    contents=prompt
)

print(response.text)

Based on your refined prompt, here is the cleaned and standardized list:

```json
["New York"]
```

### Breakdown of the steps taken:
1. **Trimmed Whitespace:** `" New York "` became `"New York"`.
2. **Title Case Applied:** `"new york"` and `"NEW YORK"` were converted to `"New York"`.
3. **Merged Names:** `"New York City"` was standardized to `"New York"`.
4. **Deduplicated:** The duplicate entries were merged into a single, clean value.


Observation: The refined prompt, which adds explicit formatting rules and a merging rule, produces a much more precise and usable result than the vague version.

# 7.8: Experiment – Persona + Format Prompting for Date Standardization

**Prompting Technique:** Persona + Format Prompting

**Formula/Template:** Role + Task + Output Format

In [16]:
prompt = """
You are a data engineer preparing a dataset for a global reporting system.

Standardize the following dates to ISO format (YYYY-MM-DD) and return them as a JSON list:
["03/08/2026", "August 3, 2026", "2026.08.03", "03-Aug-2026"]
"""

response = client.models.generate_content(
    model="gemini-3.5-flash",
    contents=prompt
)

print(response.text)

```json
[
  "2026-08-03",
  "2026-08-03",
  "2026-08-03",
  "2026-08-03"
]
```


Observation: Combining a persona with an explicit output format (JSON) produces a machine-readable, directly usable result.

# 7.9: Experiment – Step-by-Step Instruction Prompting for Currency Cleaning

**Prompting Technique:** Step-by-Step Instruction Prompting

**Formula/Template:** Numbered Instructions

In [17]:
prompt = """
Clean the following price values by:
1. Removing currency symbols and commas.
2. Converting all values to plain numeric format (float).
3. Flagging any value that cannot be converted.

Prices: ["$1,200.50", "Rs. 999", "2500", "N/A", "€350,00"]
"""

response = client.models.generate_content(
    model="gemini-3.5-flash",
    contents=prompt
)

print(response.text)

Here is the cleaned data, followed by the Python code used to process it.

### Cleaned Prices

| Original Value | Cleaned Value (Float) | Status / Flag |
| :--- | :--- | :--- |
| `"$1,200.50"` | `1200.50` | ✅ Success |
| `"Rs. 999"` | `999.0` | ✅ Success |
| `"2500"` | `2500.0` | ✅ Success |
| `"N/A"` | `None` | ❌ Failed (Flagged: Non-numeric) |
| `"€350,00"` | `350.0` | ✅ Success (Handled European decimal comma) |

---

### Python Code for the Cleaning Logic

This script automatically handles currency symbols, thousands separators (commas), and European-style decimal commas (e.g., `350,00` to `350.00`).

```python
import re

def clean_price(price_str):
    # Ensure it's a string
    price_str = str(price_str).strip()
    
    # Remove common currency symbols and letters
    # Keeps only digits, commas, dots, and minus signs
    cleaned = re.sub(r'[^\d.,-]', '', price_str)
    
    if not cleaned:
        return None, "Flagged: Non-numeric"
    
    try:
        # Handle cases with bot

Observation: Breaking the task into numbered instructions leads to a more systematic, rule-following cleanup and clear flagging of bad values.

# 7.10: Experiment – Negative Constraint Prompting for Safe Imputation

**Prompting Technique:** Negative Constraint Prompting

**Formula/Template:** Task + 'Do not...' Constraint

In [18]:
prompt = """
Suggest how to fill the missing 'salary' values below using the median of the available values.
Do not invent specific numeric values yourself — only describe the method and show the computed median from the given data.

salaries = [45000, 52000, None, 48000, None, 51000]
"""

response = client.models.generate_content(
    model="gemini-3.5-flash",
    contents=prompt
)

print(response.text)

To fill the missing `None` values using the median of the available data, you can follow this step-by-step method:

### Step-by-Step Method

1. **Extract the available (non-null) values:** 
   Filter out the `None` values from the dataset to isolate the numerical data.
   * Available values: `[45000, 52000, 48000, 51000]`

2. **Sort the available values in ascending order:**
   * Sorted list: `[45000, 48000, 51000, 52000]`

3. **Calculate the median:**
   * Since there is an **even** number of values (4 values), the median is the average of the two middle numbers (the 2nd and 3rd values):
     $$\text{Median} = \frac{48000 + 51000}{2} = \frac{99000}{2} = 49500$$
   * The computed median is **49,500**.

4. **Replace the missing values:**
   Substitute every `None` in the original list with the computed median of **49,500**.

---

### Final Result
Using this method, the completed list of salaries becomes:
`[45000, 52000, 49500, 48000, 49500, 51000]`


Observation: The negative constraint ('do not invent values') reduces the risk of the model fabricating plausible-looking but incorrect numbers.

# 7.11: Experiment – Comparative Prompting for Imputation Strategy

**Prompting Technique:** Comparative Prompting

**Formula/Template:** Task + Compare + Recommend

In [19]:
prompt = """
Compare mean imputation vs median imputation for filling missing values in this income dataset, and recommend which is more appropriate with justification:

incomes = [32000, 35000, 31000, 998000, 33000, None, 34000]
"""

response = client.models.generate_content(
    model="gemini-3.5-flash",
    contents=prompt
)

print(response.text)

To determine whether mean or median imputation is more appropriate for this dataset, we must first analyze the distribution of the existing data and calculate both metrics.

### 1. Data Analysis & Calculations
The non-missing values in your dataset are: 
`[31000, 32000, 33000, 34000, 35000, 998000]`

* **The Outlier:** There is a extreme outlier in this dataset: **`998,000`**. The other five incomes are tightly clustered between `31,000` and `35,000`.

#### **Mean Calculation (excluding the missing value):**
$$\text{Mean} = \frac{32,000 + 35,000 + 31,000 + 998,000 + 33,000 + 34,000}{6}$$
$$\text{Mean} = \frac{1,163,000}{6} \approx \mathbf{193,833.33}$$

#### **Median Calculation (excluding the missing value):**
1. Sort the values: `[31000, 32000, 33000, 34000, 35000, 998000]`
2. Because there is an even number of values (6), take the average of the two middle numbers (3rd and 4th):
$$\text{Median} = \frac{33,000 + 34,000}{2} = \mathbf{33,500}$$

---

### 2. Comparison of the Two Method

Observation: Asking the model to compare and justify (rather than just answer) surfaces the reasoning that median is more robust to the outlier (998000).

# 7.12: Experiment – Table Extraction Prompting for Column Splitting

**Prompting Technique:** Table/Extraction Prompting

**Formula/Template:** Task + Structured Input

In [20]:
prompt = """
Split the combined column below into two separate columns: "Name" and "Age".

combined_column = ["Alice, 28", "Bob, 34", "Charlie, 29"]

Return the result as a table.
"""

response = client.models.generate_content(
    model="gemini-3.5-flash",
    contents=prompt
)

print(response.text)

Here is the split data presented as a table:

| Name | Age |
| :--- | :--- |
| Alice | 28 |
| Bob | 34 |
| Charlie | 29 |


Observation: A clear extraction instruction with a defined output structure (table) yields a directly usable, correctly split result.

# 7.13: Experiment – JSON Output Prompting for Cleaned Records

**Prompting Technique:** JSON/Structured Output Prompting

**Formula/Template:** Task + Strict Schema

In [21]:
prompt = """
Clean this messy record and return ONLY valid JSON matching this schema:
{"name": string, "age": integer, "email": string}

Messy record: "  john DOE ; Age: 29yrs ; Email- JOHN.doe@MAIL.com  "
"""

response = client.models.generate_content(
    model="gemini-3.5-flash",
    contents=prompt
)

print(response.text)

{
  "name": "John Doe",
  "age": 29,
  "email": "john.doe@mail.com"
}


Observation: Specifying a strict JSON schema constrains the model to produce a parseable, standardized output rather than free-form text.

# 7.14: Experiment – Validation Prompting for Email/Phone Formats

**Prompting Technique:** Validation Prompting

**Formula/Template:** Task + Validation Criteria

In [24]:
prompt = """
Check each entry below and mark it VALID or INVALID as an email or phone number, with a short reason:

entries = ["john@mail.com", "john.mail.com", "+91-9876543210", "98765", "user@@mail.com"]
"""

response = client.models.generate_content(
    model="gemini-3.5-flash",
    contents=prompt
)

print(response.text)

Here is the evaluation for each entry:

1. **"john@mail.com"**
   * **Status:** **VALID (Email)**
   * **Reason:** It follows the standard email format containing a local part, a single "@" symbol, and a valid domain name.

2. **"john.mail.com"**
   * **Status:** **INVALID (Email)**
   * **Reason:** It is missing the mandatory "@" symbol to separate the local part from the domain.

3. **"+91-9876543210"**
   * **Status:** **VALID (Phone Number)**
   * **Reason:** It follows a valid international phone number format, including a country code (+91) and a 10-digit subscriber number.

4. **"98765"**
   * **Status:** **INVALID (Phone Number)**
   * **Reason:** It is too short to be a standard phone number (standard numbers require at least 7–15 digits).

5. **"user@@mail.com"**
   * **Status:** **INVALID (Email)**
   * **Reason:** It contains duplicate "@" symbols; a valid email must contain exactly one "@" separator.


Observation: Explicit validation criteria in the prompt lead the model to give a clear, per-item verdict useful for automated review.

# 7.15: Experiment – Explain-Then-Fix Prompting for Inconsistent Units

**Prompting Technique:** Explain-Then-Fix Prompting

**Formula/Template:** Task: Explain issue -> Fix issue

In [25]:
prompt = """
The following weight values mix units (kg and lbs) without labels:
weights = [70, 154, 65, 132, 80]

First explain how you can tell some values are likely in lbs instead of kg, then convert the likely-lbs values to kg and present a single cleaned list in kg.
"""

response = client.models.generate_content(
    model="gemini-3.5-flash",
    contents=prompt
)

print(response.text)

### How to Distinguish Between kg and lbs

To identify which values are in kilograms (kg) and which are in pounds (lbs), we can look at the distribution and scale of the numbers, assuming they represent typical adult human weights:

1. **The Scale of the Numbers:** 
   * **Kilograms:** Typical adult weights usually fall between **50 kg and 100 kg**. The values **70, 65, and 80** fit perfectly into this healthy/common adult weight range.
   * **Pounds:** Typical adult weights in pounds usually fall between **110 lbs and 220 lbs**. The values **154 and 132** fit perfectly into this range.
2. **Identifying Outliers:** 
   * If the entire list were in kilograms, 132 kg (291 lbs) and 154 kg (340 lbs) would be extreme outliers compared to the 65–80 kg group. 
   * Conversely, if the entire list were in pounds, 65 lbs (29 kg), 70 lbs (31 kg), and 80 lbs (36 kg) would be dangerously underweight for adults.
3. **Mathematical Relationship:** 
   * 154 lbs divided by 2.2 is exactly **70 kg**.
   

Observation: Separating 'explain' from 'fix' forces the model to justify its unit-inference logic before acting, making the cleaning decision auditable.

# 7.16: Experiment – Hallucination-Safety Prompting for Missing IDs

**Prompting Technique:** Hallucination-Safety Prompting

**Formula/Template:** Task + 'If unknown, say unknown'

In [26]:
prompt = """
Some customer IDs below are missing. Do not invent new ID numbers.
If an ID is missing, write "UNKNOWN" instead of guessing a value.

records = [
 {"id": "C001", "name": "Alice"},
 {"id": None, "name": "Bob"},
 {"id": "C003", "name": "Charlie"},
 {"id": None, "name": "David"}
]
"""

response = client.models.generate_content(
    model="gemini-3.5-flash",
    contents=prompt
)

print(response.text)

Here is the updated list of records with the missing IDs replaced by "UNKNOWN":

```python
records = [
 {"id": "C001", "name": "Alice"},
 {"id": "UNKNOWN", "name": "Bob"},
 {"id": "C003", "name": "Charlie"},
 {"id": "UNKNOWN", "name": "David"}
]
```


Observation: The explicit safety instruction prevents the model from fabricating plausible-looking but fake ID values.

# 7.17: Experiment – Rule-Based Prompting for Consistent Null Handling

**Prompting Technique:** Rule-Based Prompting

**Formula/Template:** Task + Explicit Rule

In [27]:
prompt = """
Apply this exact rule to the dataset below: replace every one of "N/A", "null", "-", and empty string with the value None, and leave all other values unchanged.

data = ["25", "N/A", "", "null", "-", "30", "NaN"]
"""

response = client.models.generate_content(
    model="gemini-3.5-flash",
    contents=prompt
)

print(response.text)

Here is the dataset with the rule applied:

```python
data = ["25", None, None, None, None, "30", "NaN"]
```

### Explanation of the replacements:
* `"25"` $\rightarrow$ Unchanged
* `"N/A"` $\rightarrow$ Replaced with `None`
* `""` (empty string) $\rightarrow$ Replaced with `None`
* `"null"` $\rightarrow$ Replaced with `None`
* `"-"` $\rightarrow$ Replaced with `None`
* `"30"` $\rightarrow$ Unchanged
* `"NaN"` $\rightarrow$ Unchanged (since it was not specified in the rule)


Observation: Providing one explicit rule to apply consistently avoids ambiguity about which placeholder strings count as missing.

# 7.18: Experiment – Code Generation Prompting for a Reusable Cleaning Function

**Prompting Technique:** Code Generation Prompting

**Formula/Template:** Task + Requirements

In [28]:
prompt = """
Write a Python function called clean_column(series) that takes a pandas Series of strings and:
1. Strips leading/trailing whitespace
2. Converts to Title Case
3. Replaces empty strings with None

Only return the function code, with comments explaining each step.
"""

response = client.models.generate_content(
    model="gemini-3.5-flash",
    contents=prompt
)

print(response.text)

```python
import pandas as pd


def clean_column(series: pd.Series) -> pd.Series:
    # Step 1: Strip leading/trailing whitespace and convert to Title Case.
    # We use the .str accessor to perform vectorized string operations.
    cleaned_series = series.str.strip().str.title()

    # Step 2: Replace empty strings with None.
    # After stripping, any strings that were only whitespace will become empty strings ("").
    cleaned_series = cleaned_series.replace("", None)

    return cleaned_series
```


Observation: Even though we aren't executing manual cleaning code ourselves, this shows the model can generate a reusable cleaning utility on request — useful as a starting point for a real pipeline.

# 7.19: Experiment – Data Quality Report Prompting

**Prompting Technique:** Report-Generation Prompting

**Formula/Template:** Task + Report Structure

In [29]:
prompt = """
Generate a short data quality report for the dataset below, structured as:
1. Summary of Issues Found
2. Severity (Low/Medium/High) per issue
3. Recommended Fix per issue

Dataset:
id, name, age, signup_date
1, Alice, 28, 2024-01-05
2, , -3, 2024/31/01
3, Alice, 28, 2024-01-05
4, Bob, 250,
"""

response = client.models.generate_content(
    model="gemini-3.5-flash",
    contents=prompt
)

print(response.text)

Here is the data quality report for the provided dataset.

---

### 1. Summary of Issues Found

*   **Issue A: Missing/Null Values**
    *   ID 2 is missing a `name`.
    *   ID 4 is missing a `signup_date`.
*   **Issue B: Out-of-Range/Invalid Age Values**
    *   ID 2 has a negative age (`-3`).
    *   ID 4 has an biologically impossible age (`250`).
*   **Issue C: Inconsistent and Invalid Date Formats**
    *   ID 2 uses an inconsistent and potentially invalid format (`2024/31/01` using YYYY/DD/MM with slashes) compared to the standard format (`YYYY-MM-DD`) used in ID 1 and 3.
*   **Issue D: Duplicate Records**
    *   ID 1 and ID 3 contain identical user data (Alice, 28, 2024-01-05), representing a duplicate entry under different system IDs.

---

### 2. Severity per Issue

*   **Issue A (Missing/Null Values):** **Medium** 
    *   *Reasoning:* Missing core profile fields like name and signup date limits usability and breaks downstream database constraints if those columns are confi

Observation: Requesting a structured report format turns raw observations into something that reads like an actual data-quality audit deliverable.

# 7.20: Experiment – End-to-End Cleaning Pipeline Prompting

**Prompting Technique:** Pipeline/Meta Prompting

**Formula/Template:** Task + Full Pipeline Request

In [30]:
prompt = """
Based on everything discussed in this notebook (missing values, duplicates, wrong types, outliers, inconsistent text/units, invalid formats), describe a complete step-by-step data cleaning pipeline that a data engineer should apply, in order, before this kind of dataset is used for analysis.
"""

response = client.models.generate_content(
    model="gemini-3.5-flash",
    contents=prompt
)

print(response.text)

A robust, production-ready data cleaning pipeline must be executed in a specific, logical order. Cleaning data in the wrong sequence can lead to corrupted data, skewed statistics, or wasted computational power (for example, imputing missing values on rows that are actually duplicates).

Here is the step-by-step data cleaning pipeline a data engineer should implement, ordered by logical dependency:

---

### **Phase 1: Schema Enforcement & Type Casting (Wrong Types & Invalid Formats)**
Before you can analyze or manipulate data, it must be in the correct structure. 

1. **Parse & Standardize Formats:** 
   * Identify columns with invalid or mixed formats (e.g., dates written as `MM/DD/YYYY` in some rows and `YYYY-MM-DD` in others). Standardize them to a unified format (like ISO 8601: `YYYY-MM-DD`).
   * Clean string representations of numbers (e.g., removing currency symbols like `$`, commas, or percentage signs `%` so they can be parsed as numbers).
2. **Enforce Strict Type Casting:**
 

Observation: This final meta-prompt asks the model to synthesize all previous cleaning concepts into one coherent, ordered pipeline — a good way to check overall understanding.

# Class Activity

Improve the given bad prompt into a good prompt to get a reliable data cleaning result from the LLM.

In [31]:
bad_prompt = """
Clean this data: [None, "25", "abc", "-5", "30"]
"""

In [32]:
good_prompt = """
Clean this list of age values: [None, "25", "abc", "-5", "30"]
Rules:
- Convert valid numeric strings to integers.
- Replace missing (None) values with the median of the valid ages.
- Replace non-numeric or invalid values (e.g., "abc") and impossible values (negative ages) with the median as well.
- Return the final cleaned list only.
"""

print("=== Vague Prompt ===")
response1 = client.models.generate_content(
    model="gemini-3.5-flash",
    contents=bad_prompt
)
print(response1.text)

=== Vague Prompt ===
Here is the cleaned data, along with the steps and Python code used to clean it.

### Cleaned Data
**`[25, -5, 30]`**

---

### Cleaning Criteria:
1. **Removed `None`** (null/missing value).
2. **Removed `"abc"`** (invalid non-numeric string).
3. **Converted valid numeric strings** (`"25"`, `"-5"`, `"30"`) into actual integers.

*(Note: If your use case requires only **positive** numbers, the cleaned list would be `[25, 30]`)*.

---

### Python Code to Clean This Data
If you need to do this programmatically in Python, here is the most robust way:

```python
data = [None, "25", "abc", "-5", "30"]


def clean_item(item):
    if item is None:
        return None
    try:
        # Convert valid strings to integers
        return int(item)
    except ValueError:
        # Ignore non-numeric strings like "abc"
        return None


# Filter out the None values after conversion
cleaned_data = [
    clean_item(x) for x in data if clean_item(x) is not None
]

print(cleaned

In [44]:
import time

def safe_generate(prompt, model="gemini-3.5-flash", retries=3, delay=15):
    for attempt in range(retries):
        try:
            return client.models.generate_content(model=model, contents=prompt)
        except Exception as e:
            if "RESOURCE_EXHAUSTED" in str(e) and attempt < retries - 1:
                print(f"Rate limited, waiting {delay}s...")
                time.sleep(delay)
            else:
                raise
    return None